# Environmental Factor Preview: Geemap


## 1. Dependencies and Earth Engine Initialization


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

import ee

PUBLIC_ASSET_PROJECT_ID = "summer-presence-475413-i4"
PUBLIC_ASSET_ROOT = f"projects/{PUBLIC_ASSET_PROJECT_ID}/assets"


def env_flag_enabled(name: str, *, default: bool = False) -> bool:
    value = os.environ.get(name)
    if value is None:
        return bool(default)
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on"}


PROJECT_ID = os.environ.get("EE_PROJECT_ID", "").strip()
if not PROJECT_ID:
    raise RuntimeError(
        "EE_PROJECT_ID is not set. This notebook reads public Earth Engine assets from "
        "PUBLIC_ASSET_ROOT, but Earth Engine computations should run under your own "
        "Earth Engine-enabled Google Cloud project.\n\n"
        "Set it before running this notebook:\n"
        "  macOS/Linux: export EE_PROJECT_ID='your-project-id'\n"
        "  Windows PowerShell: $env:EE_PROJECT_ID='your-project-id'"
    )

try:
    ee.Initialize(project=PROJECT_ID)
except Exception as exc:
    if not env_flag_enabled("EE_ALLOW_INTERACTIVE_AUTH"):
        raise RuntimeError(
            "Earth Engine initialization failed and interactive authentication is disabled.\n"
            "Run `earthengine authenticate`, set GOOGLE_APPLICATION_CREDENTIALS, or set "
            "`EE_ALLOW_INTERACTIVE_AUTH=1` before running this notebook interactively.\n\n"
            f"Details: {exc}"
        ) from exc
    print("Earth Engine initialization failed. Attempting interactive authentication...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("Earth Engine runtime project:", PROJECT_ID)
print("Published public asset project:", PUBLIC_ASSET_PROJECT_ID)
print("Published public asset root:", PUBLIC_ASSET_ROOT)


## 2. Workspace Paths and Study Area AOI (QTP Boundary)


In [ ]:
def bbox_from_bounds_coords(coords) -> tuple[float, float, float, float]:
    ring = coords[0]
    xs = [pt[0] for pt in ring]
    ys = [pt[1] for pt in ring]
    return (min(xs), min(ys), max(xs), max(ys))


AOI_ASSET_ID = f"{PUBLIC_ASSET_ROOT}/Qinghai-Tibet_Plateau"
aoi = ee.FeatureCollection(AOI_ASSET_ID)
region = aoi.geometry()
aoi_feature_count = int(aoi.size().getInfo())
aoi_bbox = bbox_from_bounds_coords(region.bounds(100).coordinates().getInfo())

print("AOI source:", AOI_ASSET_ID)
print("AOI feature count:", aoi_feature_count)
print("AOI bbox:", aoi_bbox)
print("AOI type:", type(region))


## 3. Parameters and Official 14-Factor Definitions


In [ ]:
TARGET_YEAR = 2016
TARGET_SCALE = 100
TARGET_PROJECTION_WKT = (
    'PROJCS["Albers",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers"],PARAMETER["false_easting",0.0],PARAMETER["false_northing",0.0],PARAMETER["central_meridian",90.0],PARAMETER["standard_parallel_1",27.5],PARAMETER["standard_parallel_2",37.5],PARAMETER["latitude_of_origin",0.0],UNIT["Meter",1.0]]'
)
TARGET_CRS = TARGET_PROJECTION_WKT
TARGET_CRS_TRANSFORM = [TARGET_SCALE, 0, 0, 0, -TARGET_SCALE, 0]
SAMPLE_SCALE = 100
EXPORT_SCALE = 100
MAX_ERROR = 1000


DEM_DATASET_ID = "CGIAR/SRTM90_V4"
FLOW_ACC_DATASET_ID = "WWF/HydroSHEDS/15ACC"
TWI_MIN_TAN_SLOPE = 1e-3
LOCAL_RELIEF_RADIUS_M = 5000
ASPECT_FLAT_THRESHOLD_DEG = 0.0
SOILGRIDS_CLAY_ASSET = "projects/soilgrids-isric/clay_mean"
SOILGRIDS_SILT_ASSET = "projects/soilgrids-isric/silt_mean"
SOILGRIDS_DEPTHS = [
    ("0-5cm", 5.0),
    ("5-15cm", 10.0),
    ("15-30cm", 15.0),
    ("30-60cm", 30.0),
    ("60-100cm", 40.0),
    ("100-200cm", 100.0),
]
SOILGRIDS_PERCENT_SCALE = 10.0
VEGETATION_TYPE_ASSET_ID = f"{PUBLIC_ASSET_ROOT}/environmental_factors/vegetation_type/QTP_Vegetation_type"
VEGETATION_TYPE_PALETTE = [
    "#ffffff",
    "#fbe5a3",
    "#ffb432",
    "#ffff4c",
    "#003c00",
    "#286400",
    "#00a000",
    "#aac800",
    "#788200",
    "#964b00",
    "#966400",
    "#0046c8",
    "#00dc82",
    "#aaf0f0",
    "#dcdcdc",
]
GIC_ASSET_ID = f"{PUBLIC_ASSET_ROOT}/environmental_factors/factor_library_v2/static/GIC_v2"
NDVI_DATASET_ID = "MODIS/061/MYD13Q1"
NDVI_SUMMER_START = "06-01"
NDVI_SUMMER_END_EXCLUSIVE = "09-01"
NDVI_SCALE_FACTOR = 0.0001
SOIL_MOISTURE_DATASET_005 = "NASA/SMAP/SPL3SMP_E/005"
SOIL_MOISTURE_DATASET_006 = "NASA/SMAP/SPL3SMP_E/006"
SOIL_MOISTURE_AM_BAND = "soil_moisture_am"
SOIL_MOISTURE_PM_BAND = "soil_moisture_pm"
SOIL_MOISTURE_SUMMER_START = "06-01"
SOIL_MOISTURE_SUMMER_END_EXCLUSIVE = "09-01"
PRECIP_DATASET_ID = "ECMWF/ERA5_LAND/DAILY_AGGR"
PRECIP_BAND_NAME = "total_precipitation_sum"
PRECIP_TO_MM = 1000.0
SMDP_DATASET_ID = "ECMWF/ERA5_LAND/DAILY_AGGR"
SMDP_BAND_NAME = "total_precipitation_sum"
SMDP_TO_MM = 1000.0
SMDP_SUMMER_START = "06-01"
SMDP_SUMMER_END_EXCLUSIVE = "09-01"
TDD_DATASET_ID = "ECMWF/ERA5_LAND/DAILY_AGGR"
WARM_SEASON_START = "06-01"
WARM_SEASON_END_EXCLUSIVE = "09-01"
HISTORICAL_CLIMATE_COLLECTION_ID = "ECMWF/ERA5_LAND/DAILY_AGGR"
HISTORICAL_TMEAN_BAND = "temperature_2m"
HISTORICAL_TMAX_BAND = "temperature_2m_max"
HISTORICAL_PRECIP_BAND = "total_precipitation_sum"
HISTORICAL_DEWPOINT_BAND = "dewpoint_temperature_2m"
HISTORICAL_WIND_U_BAND = "u_component_of_wind_10m"
HISTORICAL_WIND_V_BAND = "v_component_of_wind_10m"
TDD_BASE_TEMP_C = 0.0
print(
    {
        "target_year": TARGET_YEAR,
        "target_scale": TARGET_SCALE,
        "sample_scale": SAMPLE_SCALE,
        "export_scale": EXPORT_SCALE,
        "local_relief_radius_m": LOCAL_RELIEF_RADIUS_M,
        "vegetation_type_asset": VEGETATION_TYPE_ASSET_ID,
        "gic_asset": GIC_ASSET_ID,
        "soil_moisture_datasets": [SOIL_MOISTURE_DATASET_005, SOIL_MOISTURE_DATASET_006],
        "warm_season_window": [WARM_SEASON_START, WARM_SEASON_END_EXCLUSIVE],
    }
)


### 3.1 DEM


In [ ]:
dem_raw = ee.Image(DEM_DATASET_ID).select("elevation").clip(region)
terrain_projection = ee.Projection(TARGET_CRS, TARGET_CRS_TRANSFORM)
dem = dem_raw.resample("bilinear").reproject(terrain_projection).rename("DEM")
terrain = ee.Terrain.products(dem)
print("terrain projection:", terrain_projection.getInfo())
print("target crsTransform:", TARGET_CRS_TRANSFORM)
print("DEM band:", dem.bandNames().getInfo())


### 3.2 Slope


In [ ]:
slope = terrain.select("slope").rename("Slope")
print("Slope band:", slope.bandNames().getInfo())

### 3.3 Aspect


In [ ]:
def classify_aspect_image(
    aspect_deg_image: ee.Image,
    slope_image: ee.Image,
    mask_image: ee.Image,
    band_name: str = "Aspect",
) -> ee.Image:
    base = aspect_deg_image.multiply(0)
    aspect_nonflat = (
        base.add(2)
        .where(aspect_deg_image.gte(22.5).And(aspect_deg_image.lt(67.5)), 3)
        .where(aspect_deg_image.gte(67.5).And(aspect_deg_image.lt(112.5)), 4)
        .where(aspect_deg_image.gte(112.5).And(aspect_deg_image.lt(157.5)), 5)
        .where(aspect_deg_image.gte(157.5).And(aspect_deg_image.lt(202.5)), 6)
        .where(aspect_deg_image.gte(202.5).And(aspect_deg_image.lt(247.5)), 7)
        .where(aspect_deg_image.gte(247.5).And(aspect_deg_image.lt(292.5)), 8)
        .where(aspect_deg_image.gte(292.5).And(aspect_deg_image.lt(337.5)), 9)
    )
    non_flat = slope_image.gt(ASPECT_FLAT_THRESHOLD_DEG)
    return base.where(non_flat, aspect_nonflat).updateMask(mask_image).rename(band_name).toInt16()

aspect_deg = terrain.select("aspect").rename("Aspect_deg")
aspect = classify_aspect_image(aspect_deg, slope, dem.mask()).reproject(terrain_projection)
ASPECT_CLASS_LABELS = {
    0: "Flat",
    2: "N",
    3: "NE",
    4: "E",
    5: "SE",
    6: "S",
    7: "SW",
    8: "W",
    9: "NW",
    }
print("Aspect band:", aspect.bandNames().getInfo())
print("Aspect classes:", ASPECT_CLASS_LABELS)

### 3.4 TWI


In [ ]:
flow_acc = ee.Image(FLOW_ACC_DATASET_ID).select("b1").clip(region)
flow_acc_100m = flow_acc.resample("bilinear").reproject(terrain_projection)
cell_size_m = ee.Image.constant(TARGET_SCALE)
slope_for_twi = slope.reproject(terrain_projection)
slope_rad = slope_for_twi.multiply(3.141592653589793 / 180.0)
safe_tan_slope = slope_rad.tan().max(ee.Image.constant(TWI_MIN_TAN_SLOPE))
twi = flow_acc_100m.add(1).multiply(cell_size_m).divide(safe_tan_slope).log().rename("TWI")
print("TWI band:", twi.bandNames().getInfo())

### 3.5 Local_Relief


In [ ]:
def build_local_relief_layer(
    dem_image: ee.Image,
    radius_m: float = LOCAL_RELIEF_RADIUS_M,
    band_name: str = "Local_Relief",
    projection: ee.Projection | None = terrain_projection,
) -> ee.Image:
    kernel = ee.Kernel.circle(radius=radius_m, units="meters", normalize=False)
    neighborhood_stats = dem_image.reduceNeighborhood(reducer=ee.Reducer.minMax(), kernel=kernel)
    relief_image = neighborhood_stats.select('.*_max').subtract(neighborhood_stats.select('.*_min')).updateMask(dem_image.mask()).rename(band_name)
    if projection is not None:
        relief_image = relief_image.reproject(projection)
    return relief_image


local_relief = build_local_relief_layer(dem, radius_m=LOCAL_RELIEF_RADIUS_M, band_name="Local_Relief")
print("Local_Relief band:", local_relief.bandNames().getInfo())
print("Local_Relief radius (m):", LOCAL_RELIEF_RADIUS_M)


### 3.6 Soil


In [ ]:
def normalize_soilgrids_depth_label(depth_label: str) -> str:
    if depth_label.startswith("val_") and depth_label.endswith("_mean"):
        depth_label = depth_label[4:-5]
        return depth_label.replace("_", "-")
    return depth_label


def resolve_soilgrids_band_name(available_bands, variable_prefix: str, depth_label: str) -> str:
    normalized_depth = normalize_soilgrids_depth_label(depth_label)
    candidates = [
        f"{variable_prefix}_{normalized_depth}_mean",
        f"val_{normalized_depth.replace('-', '_')}_mean",
    ]
    for candidate in candidates:
        if candidate in available_bands:
            return candidate
    raise KeyError(
        f"{variable_prefix} has no available band at depth {normalized_depth}. Candidates: {candidates}; Available: {sorted(available_bands)}"
    )


def build_soil_fine_sediment_layer(
    region: ee.Geometry,
    projection: ee.Projection | None = terrain_projection,
):
    clay = ee.Image(SOILGRIDS_CLAY_ASSET).clip(region).resample("bilinear")
    silt = ee.Image(SOILGRIDS_SILT_ASSET).clip(region).resample("bilinear")
    clay_bands = set(clay.bandNames().getInfo())
    silt_bands = set(silt.bandNames().getInfo())

    weighted_sum = ee.Image.constant(0).toFloat()
    total_thickness_cm = 0.0
    band_pairs = []
    for depth_label, thickness_cm in SOILGRIDS_DEPTHS:
        clay_band = resolve_soilgrids_band_name(clay_bands, "clay", depth_label)
        silt_band = resolve_soilgrids_band_name(silt_bands, "silt", depth_label)
        band_pairs.append((depth_label, clay_band, silt_band))
        fine_pct = (
            clay.select(clay_band).divide(SOILGRIDS_PERCENT_SCALE)
            .add(silt.select(silt_band).divide(SOILGRIDS_PERCENT_SCALE))
        )
        weighted_sum = weighted_sum.add(fine_pct.multiply(thickness_cm))
        total_thickness_cm += thickness_cm

    soil = weighted_sum.divide(total_thickness_cm).rename("Soil")
    if projection is not None:
        soil = soil.reproject(projection)
    return soil, band_pairs


soil, soil_band_pairs = build_soil_fine_sediment_layer(region)
print("SoilGrids band pairs:", soil_band_pairs)
print("Soil band:", soil.bandNames().getInfo())

### 3.7 Vegetation_type


In [ ]:
vegetation_type = (
    ee.Image(VEGETATION_TYPE_ASSET_ID)
    .clip(region)
    .rename("Vegetation_type")
    .reproject(terrain_projection)
    .toInt16()
)
print("Vegetation_type band:", vegetation_type.bandNames().getInfo())
print("Vegetation_type source:", VEGETATION_TYPE_ASSET_ID)


### 3.8 GIC


In [ ]:
gic = (
    ee.Image(GIC_ASSET_ID)
    .clip(region)
    .resample("bilinear")
    .reproject(terrain_projection)
    .rename("GIC")
)
print("GIC band:", gic.bandNames().getInfo())


### 3.9 NDVI / Summer_SoilMoisture


In [ ]:
def build_ndvi_for_year(year: int, band_name: str = "NDVI") -> ee.Image:
    summer_start = f"{year}-{NDVI_SUMMER_START}"
    summer_end = f"{year}-{NDVI_SUMMER_END_EXCLUSIVE}"
    return (
        ee.ImageCollection(NDVI_DATASET_ID)
        .filterDate(summer_start, summer_end)
        .select("NDVI")
        .median()
        .multiply(NDVI_SCALE_FACTOR)
        .clip(region)
        .resample("bilinear")
        .reproject(terrain_projection)
        .rename(band_name)
    )

def build_soil_moisture_collection(start_date: str, end_date: str) -> ee.ImageCollection:
    return ee.ImageCollection(SOIL_MOISTURE_DATASET_005).merge(ee.ImageCollection(SOIL_MOISTURE_DATASET_006)).filterDate(start_date, end_date)

def combine_daily_soil_moisture(image: ee.Image, band_name: str = "soil_moisture_daily") -> ee.Image:
    return (
        ee.ImageCollection.fromImages([
            image.select(SOIL_MOISTURE_AM_BAND).rename(band_name),
            image.select(SOIL_MOISTURE_PM_BAND).rename(band_name),
        ])
        .mean()
        .copyProperties(image, ["system:time_start"])
    )

def build_summer_soil_moisture_for_year(year: int, band_name: str = "Summer_SoilMoisture") -> ee.Image:
    summer_start = f"{year}-{SOIL_MOISTURE_SUMMER_START}"
    summer_end = f"{year}-{SOIL_MOISTURE_SUMMER_END_EXCLUSIVE}"
    return (
        build_soil_moisture_collection(summer_start, summer_end)
        .select([SOIL_MOISTURE_AM_BAND, SOIL_MOISTURE_PM_BAND])
        .map(lambda image: combine_daily_soil_moisture(image, band_name="soil_moisture_daily"))
        .mean()
        .clip(region)
        .resample("bilinear")
        .reproject(terrain_projection)
        .rename(band_name)
    )

ndvi = build_ndvi_for_year(TARGET_YEAR, band_name="NDVI")
print("NDVI band:", ndvi.bandNames().getInfo())
print("NDVI source:", NDVI_DATASET_ID, f"{NDVI_SUMMER_START} to 08-31", "-> summer median NDVI")
summer_soil_moisture = build_summer_soil_moisture_for_year(TARGET_YEAR, band_name="Summer_SoilMoisture")
print("Summer_SoilMoisture band:", summer_soil_moisture.bandNames().getInfo())
print("Summer_SoilMoisture source:", [SOIL_MOISTURE_DATASET_005, SOIL_MOISTURE_DATASET_006], SOIL_MOISTURE_SUMMER_START, "to 08-31", "-> summer mean daily soil moisture")


### 3.10 Precipitation


In [ ]:
def build_precip_for_year(year: int, band_name: str = "Precipitation") -> ee.Image:
    year_start = f"{year}-01-01"
    year_end = f"{year + 1}-01-01"
    return (
        ee.ImageCollection(PRECIP_DATASET_ID)
        .filterDate(year_start, year_end)
        .select(PRECIP_BAND_NAME)
        .sum()
        .multiply(PRECIP_TO_MM)
        .clip(region)
        .resample("bilinear")
        .reproject(terrain_projection)
        .rename(band_name)
    )

precip = build_precip_for_year(TARGET_YEAR, band_name="Precipitation")
print("Precipitation band:", precip.bandNames().getInfo())
print("Precipitation source:", PRECIP_DATASET_ID, PRECIP_BAND_NAME, "-> mm/year")

### 3.11 SMDP


In [ ]:
def build_smdp_for_year(year: int, band_name: str = "SMDP") -> ee.Image:
    summer_start = f"{year}-{SMDP_SUMMER_START}"
    summer_end = f"{year}-{SMDP_SUMMER_END_EXCLUSIVE}"
    return (
        ee.ImageCollection(SMDP_DATASET_ID)
        .filterDate(summer_start, summer_end)
        .select(SMDP_BAND_NAME)
        .max()
        .multiply(SMDP_TO_MM)
        .clip(region)
        .resample("bilinear")
        .reproject(terrain_projection)
        .rename(band_name)
    )

smdp = build_smdp_for_year(TARGET_YEAR, band_name="SMDP")
print("SMDP band:", smdp.bandNames().getInfo())
print("SMDP source:", SMDP_DATASET_ID, SMDP_BAND_NAME, "-> mm/day, summer max daily precipitation")

### 3.12 TDD


In [ ]:
def positive_temperature_c(image: ee.Image) -> ee.Image:
    temp_c = image.subtract(273.15)
    return temp_c.max(ee.Image.constant(TDD_BASE_TEMP_C)).copyProperties(image, ["system:time_start"])


def build_tdd_for_year(year: int, band_name: str = "TDD") -> ee.Image:
    year_start = f"{year}-01-01"
    year_end = f"{year + 1}-01-01"
    return (
        ee.ImageCollection(TDD_DATASET_ID)
        .filterDate(year_start, year_end)
        .select("temperature_2m")
        .map(positive_temperature_c)
        .sum()
        .clip(region)
        .resample("bilinear")
        .reproject(terrain_projection)
        .rename(band_name)
    )


tdd_year_start = f"{TARGET_YEAR}-01-01"
tdd_year_end = f"{TARGET_YEAR + 1}-01-01"
tdd = build_tdd_for_year(TARGET_YEAR, band_name="TDD")
era5_day_count = (
    ee.ImageCollection(TDD_DATASET_ID)
    .filterDate(tdd_year_start, tdd_year_end)
    .select("temperature_2m")
    .size()
)
print("ERA5-Land day count:", era5_day_count.getInfo())
print("TDD band:", tdd.bandNames().getInfo())
print("TDD source:", TDD_DATASET_ID, "temperature_2m", "full year", "-> cumulative positive temperature in Celsius")


### 3.13 WSTmax


In [ ]:
def warm_season_window(year: int) -> tuple[str, str]:
    return (f"{year}-{WARM_SEASON_START}", f"{year}-{WARM_SEASON_END_EXCLUSIVE}")


def get_historical_era5_collection(start_date: str, end_date: str) -> ee.ImageCollection:
    return (
        ee.ImageCollection(HISTORICAL_CLIMATE_COLLECTION_ID)
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .select(
            [
                HISTORICAL_TMEAN_BAND,
                HISTORICAL_TMAX_BAND,
                HISTORICAL_PRECIP_BAND,
                HISTORICAL_DEWPOINT_BAND,
                HISTORICAL_WIND_U_BAND,
                HISTORICAL_WIND_V_BAND,
            ]
        )
    )


def build_tmax_for_year(year: int, band_name: str = "WSTmax") -> ee.Image:
    warm_start, warm_end = warm_season_window(year)
    return (
        get_historical_era5_collection(warm_start, warm_end)
        .select(HISTORICAL_TMAX_BAND)
        .max()
        .subtract(273.15)
        .clip(region)
        .resample("bilinear")
        .reproject(terrain_projection)
        .rename(band_name)
    )


tmax = build_tmax_for_year(TARGET_YEAR, band_name="WSTmax")
print("WSTmax band:", tmax.bandNames().getInfo())
print("WSTmax source:", HISTORICAL_CLIMATE_COLLECTION_ID, HISTORICAL_TMAX_BAND, f"{WARM_SEASON_START} to 08-31", "-> warm-season max daily air temperature in Celsius")


### 3.14 WSRH


In [ ]:
def calculate_relative_humidity_pct(temp_c: ee.Image, dewpoint_c: ee.Image) -> ee.Image:
    alpha = 17.625
    beta = 243.04
    saturation_ratio = temp_c.multiply(alpha).divide(temp_c.add(beta)).exp()
    actual_ratio = dewpoint_c.multiply(alpha).divide(dewpoint_c.add(beta)).exp()
    return actual_ratio.divide(saturation_ratio).multiply(100.0).clamp(0.0, 100.0)


def build_warm_season_rh_for_year(year: int, band_name: str = "WSRH") -> ee.Image:
    warm_start, warm_end = warm_season_window(year)

    def _daily_rh(image: ee.Image) -> ee.Image:
        temp_c = image.select(HISTORICAL_TMEAN_BAND).subtract(273.15)
        dewpoint_c = image.select(HISTORICAL_DEWPOINT_BAND).subtract(273.15)
        return calculate_relative_humidity_pct(temp_c, dewpoint_c).rename(band_name).copyProperties(image, ["system:time_start"])

    return (
        get_historical_era5_collection(warm_start, warm_end)
        .map(_daily_rh)
        .mean()
        .clip(region)
        .resample("bilinear")
        .reproject(terrain_projection)
        .rename(band_name)
    )


warm_season_rh_target_year = build_warm_season_rh_for_year(TARGET_YEAR, band_name="WSRH")
print("WSRH band:", warm_season_rh_target_year.bandNames().getInfo())
print("WSRH source:", HISTORICAL_CLIMATE_COLLECTION_ID, [HISTORICAL_TMEAN_BAND, HISTORICAL_DEWPOINT_BAND], f"{WARM_SEASON_START} to 08-31", "-> warm-season mean relative humidity")


## 4. Geemap Factor Preview


In [ ]:
import geemap
from ipyleaflet import WidgetControl
import ipywidgets as widgets

center = region.bounds(MAX_ERROR).centroid(MAX_ERROR).coordinates().getInfo()[::-1]
m = geemap.Map(center=center, zoom=5, basemap="ROADMAP", toolbar_ctrl=False, layer_ctrl=False)


preview_region_mask = ee.Image.constant(1).clip(region).selfMask()


def apply_preview_mask(image: ee.Image) -> ee.Image:
    return ee.Image(image).updateMask(preview_region_mask)


def classify_aspect_for_preview(
    aspect_deg_image: ee.Image,
    slope_image: ee.Image,
    mask_image: ee.Image,
    band_name: str = "Aspect",
) -> ee.Image:
    base = aspect_deg_image.multiply(0)
    aspect_nonflat = (
        base.add(2)
        .where(aspect_deg_image.gte(22.5).And(aspect_deg_image.lt(67.5)), 3)
        .where(aspect_deg_image.gte(67.5).And(aspect_deg_image.lt(112.5)), 4)
        .where(aspect_deg_image.gte(112.5).And(aspect_deg_image.lt(157.5)), 5)
        .where(aspect_deg_image.gte(157.5).And(aspect_deg_image.lt(202.5)), 6)
        .where(aspect_deg_image.gte(202.5).And(aspect_deg_image.lt(247.5)), 7)
        .where(aspect_deg_image.gte(247.5).And(aspect_deg_image.lt(292.5)), 8)
        .where(aspect_deg_image.gte(292.5).And(aspect_deg_image.lt(337.5)), 9)
    )
    non_flat = slope_image.gt(ASPECT_FLAT_THRESHOLD_DEG)
    return base.where(non_flat, aspect_nonflat).updateMask(mask_image).rename(band_name).toInt16()


def build_preview_precip_for_year(year: int, band_name: str = "Precipitation") -> ee.Image:
    year_start = f"{year}-01-01"
    year_end = f"{year + 1}-01-01"
    collection = (
        ee.ImageCollection(PRECIP_DATASET_ID)
        .filterBounds(region)
        .filterDate(year_start, year_end)
        .select(PRECIP_BAND_NAME)
    )
    native_projection = ee.Image(collection.first()).select(PRECIP_BAND_NAME).projection()
    return collection.sum().multiply(PRECIP_TO_MM).clip(region).rename(band_name).setDefaultProjection(native_projection)


def build_preview_smdp_for_year(year: int, band_name: str = "SMDP") -> ee.Image:
    summer_start = f"{year}-{SMDP_SUMMER_START}"
    summer_end = f"{year}-{SMDP_SUMMER_END_EXCLUSIVE}"
    collection = (
        ee.ImageCollection(SMDP_DATASET_ID)
        .filterBounds(region)
        .filterDate(summer_start, summer_end)
        .select(SMDP_BAND_NAME)
    )
    native_projection = ee.Image(collection.first()).select(SMDP_BAND_NAME).projection()
    return collection.max().multiply(SMDP_TO_MM).clip(region).rename(band_name).setDefaultProjection(native_projection)


def build_preview_tdd_for_year(year: int, band_name: str = "TDD") -> ee.Image:
    year_start = f"{year}-01-01"
    year_end = f"{year + 1}-01-01"
    collection = (
        ee.ImageCollection(TDD_DATASET_ID)
        .filterBounds(region)
        .filterDate(year_start, year_end)
        .select("temperature_2m")
    )
    native_projection = ee.Image(collection.first()).select("temperature_2m").projection()
    return (
        collection.map(positive_temperature_c)
        .sum()
        .clip(region)
        .rename(band_name)
        .setDefaultProjection(native_projection)
    )


def build_preview_tmax_for_year(year: int, band_name: str = "WSTmax") -> ee.Image:
    warm_start, warm_end = warm_season_window(year)
    collection = get_historical_era5_collection(warm_start, warm_end).select(HISTORICAL_TMAX_BAND)
    native_projection = ee.Image(collection.first()).select(HISTORICAL_TMAX_BAND).projection()
    return collection.max().subtract(273.15).clip(region).rename(band_name).setDefaultProjection(native_projection)


def build_preview_warm_season_rh_for_year(year: int, band_name: str = "WSRH") -> ee.Image:
    warm_start, warm_end = warm_season_window(year)
    collection = get_historical_era5_collection(warm_start, warm_end)
    native_projection = ee.Image(collection.first()).select(HISTORICAL_TMEAN_BAND).projection()

    def _daily_rh(image: ee.Image) -> ee.Image:
        temp_c = image.select(HISTORICAL_TMEAN_BAND).subtract(273.15)
        dewpoint_c = image.select(HISTORICAL_DEWPOINT_BAND).subtract(273.15)
        return calculate_relative_humidity_pct(temp_c, dewpoint_c).rename(band_name).copyProperties(image, ["system:time_start"])

    return collection.map(_daily_rh).mean().clip(region).rename(band_name).setDefaultProjection(native_projection)


def build_preview_ndvi_for_year(year: int, band_name: str = "NDVI") -> ee.Image:
    summer_start = f"{year}-{NDVI_SUMMER_START}"
    summer_end = f"{year}-{NDVI_SUMMER_END_EXCLUSIVE}"
    collection = ee.ImageCollection(NDVI_DATASET_ID).filterDate(summer_start, summer_end).select("NDVI")
    native_projection = ee.Image(collection.first()).select("NDVI").projection()
    return collection.median().multiply(NDVI_SCALE_FACTOR).clip(region).rename(band_name).setDefaultProjection(native_projection)


def build_preview_summer_soil_moisture_for_year(year: int, band_name: str = "Summer_SoilMoisture") -> ee.Image:
    summer_start = f"{year}-{SOIL_MOISTURE_SUMMER_START}"
    summer_end = f"{year}-{SOIL_MOISTURE_SUMMER_END_EXCLUSIVE}"
    collection = build_soil_moisture_collection(summer_start, summer_end).select([SOIL_MOISTURE_AM_BAND, SOIL_MOISTURE_PM_BAND])
    native_projection = ee.Image(collection.first()).select(SOIL_MOISTURE_AM_BAND).projection()
    return (
        collection.map(lambda image: combine_daily_soil_moisture(image, band_name="soil_moisture_daily"))
        .mean()
        .clip(region)
        .rename(band_name)
        .setDefaultProjection(native_projection)
    )


terrain_preview = ee.Terrain.products(dem_raw)
dem_preview = dem_raw.rename("DEM")
slope_preview = terrain_preview.select("slope").rename("Slope")
aspect_preview = classify_aspect_for_preview(terrain_preview.select("aspect"), slope_preview, dem_raw.mask())
flow_acc_preview = ee.Image(FLOW_ACC_DATASET_ID).select("b1").clip(region)
cell_size_m_preview = slope_preview.multiply(0).add(TARGET_SCALE)
slope_rad_preview = slope_preview.multiply(3.141592653589793 / 180.0)
safe_tan_slope_preview = slope_rad_preview.tan().max(slope_preview.multiply(0).add(TWI_MIN_TAN_SLOPE))
twi_preview = flow_acc_preview.add(1).multiply(cell_size_m_preview).divide(safe_tan_slope_preview).log().rename("TWI")
local_relief_preview = build_local_relief_layer(dem_preview, radius_m=LOCAL_RELIEF_RADIUS_M, band_name="Local_Relief", projection=None)
soil_preview = build_soil_fine_sediment_layer(region, projection=None)[0].rename("Soil")
vegetation_type_preview = ee.Image(VEGETATION_TYPE_ASSET_ID).clip(region).rename("Vegetation_type")
vegetation_type_preview = vegetation_type_preview.updateMask(vegetation_type_preview.gte(0).And(vegetation_type_preview.lte(14)))
gic_preview = ee.Image(GIC_ASSET_ID).clip(region).rename("GIC")
precip_preview = build_preview_precip_for_year(TARGET_YEAR, band_name="Precipitation")
smdp_preview = build_preview_smdp_for_year(TARGET_YEAR, band_name="SMDP")
tdd_preview = build_preview_tdd_for_year(TARGET_YEAR, band_name="TDD")
tmax_preview = build_preview_tmax_for_year(TARGET_YEAR, band_name="WSTmax")
warm_season_rh_preview = build_preview_warm_season_rh_for_year(TARGET_YEAR, band_name="WSRH")
ndvi_preview = build_preview_ndvi_for_year(TARGET_YEAR, band_name="NDVI")
summer_soil_moisture_preview = build_preview_summer_soil_moisture_for_year(TARGET_YEAR, band_name="Summer_SoilMoisture")

preview_layers = [
    ("Slope", slope_preview, {"min": 0, "max": 90, "palette": ["#ebeae9", "#835538", "#8c0d01", "#6e9634", "#eefcb3", "#9dd9f1", "#61c4b1", "#325b95", "#253865", "#000000"]}, True),
    ("Aspect", aspect_preview, {"min": 0, "max": 9, "palette": ["#f3f4f6", "#f3f4f6", "#1d4ed8", "#06b6d4", "#14b8a6", "#22c55e", "#eab308", "#f97316", "#ef4444", "#a855f7"]}, False),
    ("TWI", twi_preview, {"min": 0, "max": 30, "palette": ["#fef2f3", "#efd3c0", "#c1b299", "#94a298", "#6e95a1", "#4d81a3", "#36679d", "#2a4c8f", "#232f7c", "#1a0c64"]}, False),
    ("Local_Relief", local_relief_preview, {"min": 0, "max": 5000, "palette": ["#543005", "#91560d", "#c58e3d", "#e7cf94", "#f6eed9", "#ddefed", "#98d7cd", "#47a49b", "#086d65", "#003c30"]}, False),
    ("Soil", soil_preview, {"min": 0, "max": 100, "palette": ["#000000", "#23160e", "#452c1c", "#69422a", "#8c5838", "#ae6e46", "#d28555", "#f59b63", "#ffb170", "#ffc77f"]}, False),
    ("Vegetation_type", vegetation_type_preview, {"min": 0, "max": 14, "palette": VEGETATION_TYPE_PALETTE}, False),
    ("GIC", gic_preview, {"min": 0, "max": 65, "palette": ["#f7fbff", "#e1edf8", "#ccdff1", "#abd0e6", "#82bbdb", "#5aa2cf", "#3787c0", "#1c6ab0", "#084e98", "#08306b"]}, False),
    ("Precipitation", precip_preview, {"min": 0, "max": 6000, "palette": ["#edf8fb", "#bfd3e6", "#9ebcda", "#8c96c6", "#8c6bb1", "#88419d", "#6e016b"]}, False),
    ("SMDP", smdp_preview, {"min": 0, "max": 120, "palette": ["#bd925a", "#d6bd8d", "#edeac2", "#b5c8b8", "#79a7ac", "#2887a1"]}, False),
    ("TDD", tdd_preview, {"min": 0, "max": 5000, "palette": ["#f1eef6", "#d0d1e6", "#a6bddb", "#74a9cf", "#2b8cbe", "#045a8d"]}, False),
    ("WSTmax", tmax_preview, {"min": 0, "max": 40, "palette": ["#e6f5b2", "#c0e6b5", "#7ecdbb", "#44b7c4", "#2296c1", "#216aad", "#24419a", "#13266f"]}, False),
    ("WSRH", warm_season_rh_preview, {"min": 20, "max": 100, "palette": ["#e2e2ef", "#cdcde4", "#b2b2d5", "#9995c6", "#807cba", "#6c55a5", "#5a3294", "#470f84"]}, False),
    ("NDVI", ndvi_preview, {"min": 0, "max": 0.8, "palette": ["#f7fcf5", "#74c476", "#00441b"]}, False),
    ("Summer_SoilMoisture", summer_soil_moisture_preview, {"min": 0, "max": 0.7, "palette": ["#fff3b4", "#fedb80", "#feb643", "#f78921", "#dd5f0c", "#b03f03", "#792b05"]}, False),
]

aoi_layer_spec = {
    "ee_object": ee.FeatureCollection([ee.Feature(region)]).style(color="black", fillColor="00000000", width=2),
    "vis_params": {},
    "layer_name": "AOI",
}

aspect_legend_items = [
    ("Flat (0)", "#f3f4f6"),
    ("N (2)", "#1d4ed8"),
    ("NE (3)", "#06b6d4"),
    ("E (4)", "#14b8a6"),
    ("SE (5)", "#22c55e"),
    ("S (6)", "#eab308"),
    ("SW (7)", "#f97316"),
    ("W (8)", "#ef4444"),
    ("NW (9)", "#a855f7"),
]
vegetation_type_labels = [
    "Bare land",
    "Alpine scree",
    "Alpine grassland",
    "Alpine meadow",
    "Evergreen needle-leaved forest",
    "Deciduous needle-leaved forest",
    "Evergreen broadleaved forest",
    "Deciduous broadleaved forest",
    "Mixed forest",
    "Evergreen shrubland",
    "Deciduous shrubland",
    "Water body",
    "Wetland",
    "Cultivated vegetation",
    "Permanent ice and snow",
]
vegetation_type_legend_items = list(zip(vegetation_type_labels, VEGETATION_TYPE_PALETTE))

theme_catalog = {
    "slope": {"short_label": "Slope", "label": "Slope", "ee_object": slope_preview, "vis_params": {"min": 0, "max": 45, "palette": ["#ebeae9", "#835538", "#8c0d01", "#6e9634", "#eefcb3", "#9dd9f1", "#61c4b1", "#325b95", "#253865", "#000000"]}, "layer_name": "Slope"},
    "aspect": {"short_label": "Aspect", "label": "Aspect", "ee_object": aspect_preview, "vis_params": {"min": 0, "max": 9, "palette": ["#f3f4f6", "#f3f4f6", "#1d4ed8", "#06b6d4", "#14b8a6", "#22c55e", "#eab308", "#f97316", "#ef4444", "#a855f7"]}, "layer_name": "Aspect", "legend_items": aspect_legend_items},
    "twi": {"short_label": "TWI", "label": "TWI", "ee_object": twi_preview, "vis_params": {"min": 0, "max": 30, "palette": ["#fef2f3", "#efd3c0", "#c1b299", "#94a298", "#6e95a1", "#4d81a3", "#36679d", "#2a4c8f", "#232f7c", "#1a0c64"]}, "layer_name": "TWI"},
    "local_relief": {"short_label": "Local_Relief", "label": "Local_Relief", "ee_object": local_relief_preview, "vis_params": {"min": 0, "max": 5000, "palette": ["#543005", "#91560d", "#c58e3d", "#e7cf94", "#f6eed9", "#ddefed", "#98d7cd", "#47a49b", "#086d65", "#003c30"]}, "layer_name": "Local_Relief"},
    "soil": {"short_label": "Soil", "label": "Soil", "ee_object": soil_preview, "vis_params": {"min": 0, "max": 100, "palette": ["#000000", "#23160e", "#452c1c", "#69422a", "#8c5838", "#ae6e46", "#d28555", "#f59b63", "#ffb170", "#ffc77f"]}, "layer_name": "Soil"},
    "vegetation_type": {"short_label": "Vegetation_type", "label": "Vegetation_type", "ee_object": vegetation_type_preview, "vis_params": {"min": 0, "max": 14, "palette": VEGETATION_TYPE_PALETTE}, "layer_name": "Vegetation_type", "legend_items": vegetation_type_legend_items},
    "gic": {"short_label": "GIC", "label": "GIC", "ee_object": gic_preview, "vis_params": {"min": 0, "max": 65, "palette": ["#f7fbff", "#e1edf8", "#ccdff1", "#abd0e6", "#82bbdb", "#5aa2cf", "#3787c0", "#1c6ab0", "#084e98", "#08306b"]}, "layer_name": "GIC"},
    "precip": {"short_label": "Precipitation", "label": "Precipitation", "annual_builder": lambda year: build_preview_precip_for_year(int(year), band_name="Precipitation"), "vis_params": {"min": 0, "max": 6000, "palette": ["#edf8fb", "#bfd3e6", "#9ebcda", "#8c96c6", "#8c6bb1", "#88419d", "#6e016b"]}, "layer_name": "Precipitation"},
    "smdp": {"short_label": "SMDP", "label": "SMDP", "annual_builder": lambda year: build_preview_smdp_for_year(int(year), band_name="SMDP"), "vis_params": {"min": 0, "max": 120, "palette": ["#bd925a", "#d6bd8d", "#edeac2", "#b5c8b8", "#79a7ac", "#2887a1"]}, "layer_name": "SMDP"},
    "tdd": {"short_label": "TDD", "label": "TDD", "annual_builder": lambda year: build_preview_tdd_for_year(int(year), band_name="TDD"), "vis_params": {"min": 0, "max": 5000, "palette": ["#f1eef6", "#d0d1e6", "#a6bddb", "#74a9cf", "#2b8cbe", "#045a8d"]}, "layer_name": "TDD"},
    "tmax": {"short_label": "WSTmax", "label": "WSTmax", "annual_builder": lambda year: build_preview_tmax_for_year(int(year), band_name="WSTmax"), "vis_params": {"min": 0, "max": 40, "palette": ["#e6f5b2", "#c0e6b5", "#7ecdbb", "#44b7c4", "#2296c1", "#216aad", "#24419a", "#13266f"]}, "layer_name": "WSTmax"},
    "warm_season_rh": {"short_label": "WSRH", "label": "WSRH", "annual_builder": lambda year: build_preview_warm_season_rh_for_year(int(year), band_name="WSRH"), "vis_params": {"min": 20, "max": 100, "palette": ["#e2e2ef", "#cdcde4", "#b2b2d5", "#9995c6", "#807cba", "#6c55a5", "#5a3294", "#470f84"]}, "layer_name": "WSRH"},
    "ndvi": {"short_label": "NDVI", "label": "NDVI", "annual_builder": lambda year: build_preview_ndvi_for_year(int(year), band_name="NDVI"), "vis_params": {"min": 0, "max": 0.8, "palette": ["#f7fcf5", "#74c476", "#00441b"]}, "layer_name": "NDVI"},
    "summer_soil_moisture": {"short_label": "Summer_SoilMoisture", "label": "Summer_SoilMoisture", "annual_builder": lambda year: build_preview_summer_soil_moisture_for_year(int(year), band_name="Summer_SoilMoisture"), "vis_params": {"min": 0, "max": 0.7, "palette": ["#fff3b4", "#fedb80", "#feb643", "#f78921", "#dd5f0c", "#b03f03", "#792b05"]}, "layer_name": "Summer_SoilMoisture"},
}

theme_order = [
    "slope",
    "aspect",
    "twi",
    "local_relief",
    "soil",
    "vegetation_type",
    "gic",
    "precip",
    "smdp",
    "tdd",
    "tmax",
    "warm_season_rh",
    "ndvi",
    "summer_soil_moisture",
]
PREVIEW_ANNUAL_YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022]

theme_options = [(theme_catalog[key]["short_label"], key) for key in theme_order]
managed_layer_names = [aoi_layer_spec["layer_name"], *[theme_catalog[key]["layer_name"] for key in theme_order]]
basemap_options = [
    ("Roadmap", "ROADMAP"),
    ("Satellite", "SATELLITE"),
    ("Terrain", "TERRAIN"),
]


def is_annual_theme(theme: dict) -> bool:
    return callable(theme.get("annual_builder"))


def get_theme_image(theme: dict, year: int):
    if is_annual_theme(theme):
        return theme["annual_builder"](int(year))
    return theme["ee_object"]


def current_layer_status(theme: dict, year: int) -> str:
    text = f"Current layer: {theme['label']}"
    if is_annual_theme(theme):
        text += f" | Year: {int(year)}"
    return text


def update_year_dropdown_state() -> None:
    annual = is_annual_theme(theme_catalog[theme_dropdown.value])
    year_dropdown.disabled = not annual
    year_dropdown.style.description_width = "0px"
    year_dropdown.layout.visibility = "visible"


def css_color(color: str) -> str:
    if isinstance(color, str) and not color.startswith("#"):
        if all(ch in "0123456789abcdefABCDEF" for ch in color) and len(color) in (3, 6, 8):
            return f"#{color}"
    return color


def build_legend_html(theme: dict) -> str:
    title = theme["label"]
    legend_items = theme.get("legend_items")
    if legend_items:
        rows = "".join(
            f"<div style='display:flex;align-items:flex-start;margin:1px 0;'>"
            f"<span style='display:inline-block;width:10px;height:10px;flex:0 0 10px;margin:2px 5px 0 0;border:1px solid #999;background:{css_color(color)};'></span>"
            f"<span style='font-size:10px;line-height:1.2;color:#1f2933;max-width:164px;word-break:break-word;'>{label}</span>"
            f"</div>"
            for label, color in legend_items
        )
        return (
            f"<div style='display:inline-block;max-width:188px;font-size:10px;line-height:1.2;'>"
            f"<div style='font-size:11px;font-weight:700;text-align:center;margin-bottom:2px;color:#111827;'>{title}</div>"
            f"{rows}"
            f"</div>"
        )

    vis_params = theme["vis_params"]
    palette = [css_color(color) for color in vis_params.get("palette", ["#000000", "#ffffff"])]
    gradient = ", ".join(palette)
    min_label = vis_params.get("min", "")
    max_label = vis_params.get("max", "")
    return (
        f"<div style='display:inline-block;max-width:188px;font-size:10px;line-height:1.2;'>"
        f"<div style='font-size:11px;font-weight:700;text-align:center;margin-bottom:2px;color:#111827;'>{title}</div>"
        f"<div style='width:156px;height:10px;border:1px solid #9ca3af;border-radius:999px;background:linear-gradient(to right, {gradient});'></div>"
        f"<div style='display:flex;justify-content:space-between;font-size:10px;color:#4b5563;margin-top:2px;'>"
        f"<span>{min_label}</span><span>{max_label}</span>"
        f"</div>"
        f"</div>"
    )


def remove_managed_layer(layer_name: str) -> None:
    layer_object = None
    ee_layers = getattr(m, "ee_layers", None)
    if isinstance(ee_layers, dict):
        layer_object = ee_layers.get(layer_name, {}).get("ee_layer")
    if layer_object is None:
        for candidate in tuple(m.layers):
            if getattr(candidate, "name", None) == layer_name:
                layer_object = candidate
                break
    if layer_object is not None:
        try:
            m.remove(layer_object)
        except Exception:
            pass
    if isinstance(ee_layers, dict):
        ee_layers.pop(layer_name, None)
    ee_raster_layers = getattr(m, "ee_raster_layers", None)
    if isinstance(ee_raster_layers, dict):
        ee_raster_layers.pop(layer_name, None)
    ee_vector_layers = getattr(m, "ee_vector_layers", None)
    if isinstance(ee_vector_layers, dict):
        ee_vector_layers.pop(layer_name, None)


current_theme_key_holder = {"value": None}
current_theme_year_holder = {"value": None}
aoi_layer_initialized = {"value": False}


def render_preview(_=None):
    theme_key = theme_dropdown.value
    theme = theme_catalog[theme_key]
    selected_year = int(year_dropdown.value)
    previous_theme_key = current_theme_key_holder["value"]
    previous_theme_year = current_theme_year_holder["value"]

    if not aoi_layer_initialized["value"]:
        m.addLayer(aoi_layer_spec["ee_object"], aoi_layer_spec["vis_params"], aoi_layer_spec["layer_name"], shown=True, opacity=1.0)
        aoi_layer_initialized["value"] = True

    layer_needs_refresh = previous_theme_key != theme_key or (is_annual_theme(theme) and previous_theme_year != selected_year)
    if layer_needs_refresh:
        if previous_theme_key is not None:
            remove_managed_layer(theme_catalog[previous_theme_key]["layer_name"])
        m.addLayer(apply_preview_mask(get_theme_image(theme, selected_year)), theme["vis_params"], theme["layer_name"], shown=True, opacity=1.0)
        current_theme_key_holder["value"] = theme_key
        current_theme_year_holder["value"] = selected_year

    update_year_dropdown_state()
    current_theme_html.value = (
        f"<div style='font-size:10px;line-height:1.15;color:#4b5563;margin-top:2px;'>{current_layer_status(theme, selected_year)}</div>"
    )
    legend_html.value = build_legend_html(theme)


def on_basemap_change(change):
    if change.get("name") != "value":
        return
    m._replace_basemap(change["new"])
    render_preview()


def on_theme_change(change):
    if change.get("name") != "value":
        return
    update_year_dropdown_state()
    render_preview()


def on_year_change(change):
    if change.get("name") != "value":
        return
    render_preview()


basemap_dropdown = widgets.Dropdown(
    options=basemap_options,
    value="ROADMAP",
    layout=widgets.Layout(width="170px", margin="0 auto"),
    tooltip="Choose a basemap",
)
theme_dropdown = widgets.Dropdown(
    options=theme_options,
    value="slope",
    layout=widgets.Layout(width="170px", margin="0 auto"),
    tooltip="Choose one thematic layer",
)
year_dropdown = widgets.Dropdown(
    options=[(str(year), year) for year in PREVIEW_ANNUAL_YEARS],
    value=TARGET_YEAR,
    layout=widgets.Layout(width="170px", margin="0 auto"),
    tooltip="Choose one annual layer year",
)
current_theme_html = widgets.HTML()
legend_html = widgets.HTML()


theme_dropdown.observe(on_theme_change, names="value")
year_dropdown.observe(on_year_change, names="value")
basemap_dropdown.observe(on_basemap_change, names="value")


update_year_dropdown_state()
render_preview()

selector_panel = widgets.VBox(
    [
        widgets.HTML(value="<div style='font-size:11px;font-weight:700;line-height:1.1;color:#475569;'>Basemap</div>"),
        basemap_dropdown,
        widgets.HTML(value="<div style='font-size:11px;font-weight:700;line-height:1.1;color:#475569;margin-top:4px;'>Layer</div>"),
        theme_dropdown,
        widgets.HTML(value="<div style='font-size:11px;font-weight:700;line-height:1.1;color:#475569;margin-top:4px;'>Year</div>"),
        year_dropdown,
        current_theme_html,
    ],
    layout=widgets.Layout(width="200px", padding="5px 7px", border="1px solid #d0d7de", overflow="hidden"),
)
legend_panel = widgets.VBox(
    [legend_html],
    layout=widgets.Layout(width="auto", padding="4px 6px", border="1px solid #d0d7de"),
)

m.add(WidgetControl(widget=selector_panel, position="topright"))
m.add(WidgetControl(widget=legend_panel, position="bottomright"))
m.add_layer_control(position="bottomleft")
m
